<a href="https://colab.research.google.com/github/dimamagdenko07-spec/python-data-analysis-olist/blob/main/notebooks/python_lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Задание 1

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import os

'''Чтение файлов'''
customers = pd.read_csv("olist_customers_dataset.csv")
items = pd.read_csv("olist_order_items_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
products = pd.read_csv('olist_products_dataset.csv')
'''Склеиваю данные в один DataFrame'''
df = orders.merge(items, on='order_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(products, on='product_id', how='left')

#display(df)
df.info(memory_usage='deep')
'''Меняем типы у нужных строк на category'''
categories = ['customer_state', 'order_status', 'product_category_name', 'customer_city']
df[categories] = df[categories].astype('category')
'''Преобразуем даты к типу datetime64[ns]'''
time = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date'
]
for date in time:
  df[date] = pd.to_datetime(df[date])
#print(df['order_delivered_carrier_date'].dtype)
df.info(memory_usage='deep')
print(f"До: 122 MB, После: 59.4 MB. Экономия - {100 - round(59.4/122*100, 0)}%")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       113425 non-null  object 
 1   customer_id                    113425 non-null  object 
 2   order_status                   113425 non-null  object 
 3   order_purchase_timestamp       113425 non-null  object 
 4   order_approved_at              113264 non-null  object 
 5   order_delivered_carrier_date   111457 non-null  object 
 6   order_delivered_customer_date  110196 non-null  object 
 7   order_estimated_delivery_date  113425 non-null  object 
 8   order_item_id                  112650 non-null  float64
 9   product_id                     112650 non-null  object 
 10  seller_id                      112650 non-null  object 
 11  shipping_limit_date            112650 non-null  object 
 12  price                         

Задание 2

In [ ]:
'''Строим пайплайн для чисел: заполнение пропусков + нормализация'''
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
num_cols = make_column_selector(dtype_include = 'number')
'''Строим пайплайн для строковых данных: заполнение пропусков + one-hot encoder'''
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Nothing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
cat_cols = make_column_selector(dtype_include = 'category')
'''Собираем все пайплайны в один объект'''
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('str', cat_transformer, cat_cols)
])
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])
full_pipeline.set_output(transform='pandas')
'''Проверяем наш препроцессинг'''
df_sample = df.head(100)
transformed = full_pipeline.fit_transform(df_sample)
display(transformed)
'''feature_names = full_pipeline.named_steps['preprocessor'].get_feature_names_out()
df_res = pd.DataFrame(transformed.toarray(), columns=feature_names)
display(df_res.head())'''
df['year'] = df['order_purchase_timestamp'].dt.year

,num__order_item_id,num__price,num__freight_value,num__customer_zip_code_prefix,num__product_name_lenght,num__product_description_lenght,num__product_photos_qty,num__product_weight_g,num__product_length_cm,num__product_height_cm,...,str__product_category_name_livros_interesse_geral,str__product_category_name_malas_acessorios,str__product_category_name_moveis_decoracao,str__product_category_name_moveis_escritorio,str__product_category_name_papelaria,str__product_category_name_perfumaria,str__product_category_name_pet_shop,str__product_category_name_relogios_presentes,str__product_category_name_telefonia,str__product_category_name_utilidades_domesticas
0,-0.352941,-0.513295,-0.789904,-1.126699,-0.944314,-0.864555,1.091849,-0.436543,-0.609539,-0.704152,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.352941,0.092614,0.153923,0.294603,-1.993553,-1.015297,-0.812539,-0.460591,-0.609539,-0.242715,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-0.352941,0.374019,-0.084050,1.168184,-0.372003,-0.924852,-0.812539,-0.455782,-0.341020,0.311008,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-0.352941,-0.410773,0.452398,0.660017,0.868006,-0.529573,0.457053,-0.448567,-0.018796,-0.519577,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,-0.352941,-0.582212,-0.789904,-0.934303,-1.135085,-0.784159,1.091849,-0.496663,1.108986,-0.058141,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,-0.352941,-0.410773,-0.428238,1.048533,-1.707397,-0.341983,-0.177743,-0.532734,-0.770651,-1.165588,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
96,-0.352941,-0.320614,-0.785870,-0.232307,-0.467388,0.823754,-0.812539,-0.536823,-0.931763,-0.704152,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
97,-0.352941,0.039953,0.053759,1.766121,0.200309,-0.621693,-0.812539,-0.135947,0.625650,-0.427290,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98,-0.352941,-0.455170,0.293749,0.637868,0.295694,-0.939926,-0.812539,-0.328329,0.786762,-0.058141,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Задание 3

In [ ]:
%%timeit -n 100 -r 10
'''Фильтр через маски'''
mask = (
    (df['customer_state'].isin(['SP', 'RJ'])) &
    (df['price'] > 150) &
    (df['year'] == 2018)
)
res_mask = df[mask]

4.75 ms ± 192 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [ ]:
%%timeit -n 100 -r 10
'''Фильтр через запрос'''
res_query = df.query("customer_state in ['SP', 'RJ'] and price > 150 and year == 2018")

10.7 ms ± 976 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


Задание 4

In [ ]:
'''Сравнение Parquet и Csv'''
df.to_parquet("df.parquet")
df.to_csv("df.csv")
size_csv = os.path.getsize('df.csv') / (1024 * 1024)
size_parquet = os.path.getsize('df.parquet') / (1024 * 1024)
print(f"Размер CSV: {size_csv} MB")
print(f"Размер Parquet: {size_parquet} MB")

Размер CSV: 41.37502956390381 MB
Размер Parquet: 18.08778953552246 MB


In [ ]:
%%timeit -n 7 -r 5
df_csv = pd.read_csv("df.csv")

660 ms ± 21.1 ms per loop (mean ± std. dev. of 5 runs, 7 loops each)


In [ ]:
%%timeit -n 7 -r 5
df_parquet = pd.read_parquet("df.parquet")

115 ms ± 10.3 ms per loop (mean ± std. dev. of 5 runs, 7 loops each)


Сsv хранит все данные в строках, поэтому при чтении у нас всё преобразуется в Object. Parquet хранит данные по колонкам, учитывая типы в бинарном формате, из-за чего его не нужно переводить компьютеру, а при чтении всё переводится в нужные типы, из-за чего нам не требуется создавать тяжелые Object лишний раз